# EDGE AI CHALLENGE 2026 – Warm-up
## Notebook 01: Khám Phá Dữ Liệu Object Detection (EDA)

Notebook này thực hiện phân tích và khám phá tập dữ liệu phát hiện vật thể giao thông (Object Detection):
1. Kiểm tra cấu trúc ảnh và nhãn chuẩn YOLO.
2. Tìm hiểu ý nghĩa định dạng nhãn (`class_id cx cy w h`).
3. Chuyển đổi tọa độ chuẩn hóa sang tọa độ pixel và vẽ trực quan hóa bounding box.
4. Thống kê phân bố các lớp đối tượng và các ảnh không chứa vật thể (nhãn rỗng).


### 1. Import các thư viện cần thiết

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Thiết lập hiển thị đồ thị inline trong Jupyter
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11


### 2. Khai báo đường dẫn thư mục dữ liệu

In [ ]:
# Xác định đường dẫn gốc của project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

TRAIN_IMAGES_DIR = DATA_RAW / 'train' / 'images'
TRAIN_LABELS_DIR = DATA_RAW / 'train' / 'labels'
TEST_IMAGES_DIR = DATA_RAW / 'test' / 'images'
CLASSES_PATH = DATA_RAW / 'classes.txt'

print(f"Thư mục gốc: {PROJECT_ROOT}")
print(f"Dữ liệu thô: {DATA_RAW}")


### 3. Đọc danh sách 10 lớp đối tượng (`classes.txt`)

In [ ]:
with open(CLASSES_PATH, 'r', encoding='utf-8') as f:
    classes = [line.strip() for line in f if line.strip()]

print(f"Tổng số lớp đối tượng: {len(classes)}\n")
for idx, cls_name in enumerate(classes):
    print(f"  Lớp {idx}: {cls_name}")


### 4. Đếm số lượng ảnh train, nhãn train và ảnh test

In [ ]:
image_extensions = {'.png', '.jpg', '.jpeg'}
train_images = sorted([f for f in TRAIN_IMAGES_DIR.iterdir() if f.suffix.lower() in image_extensions])
train_labels = sorted([f for f in TRAIN_LABELS_DIR.iterdir() if f.suffix.lower() == '.txt'])
test_images = sorted([f for f in TEST_IMAGES_DIR.iterdir() if f.suffix.lower() in image_extensions])

print(f"Số ảnh huấn luyện (Train images): {len(train_images)}")
print(f"Số file nhãn (Train labels):     {len(train_labels)}")
print(f"Số ảnh kiểm tra (Test images):    {len(test_images)}")


### 5 & 6. Chọn một ảnh train và hiển thị ảnh gốc

In [ ]:
# Chọn một ảnh mẫu (ví dụ ảnh '0147.png' hoặc ảnh đầu tiên)
sample_stem = '0147' if (TRAIN_IMAGES_DIR / '0147.png').exists() else train_images[0].stem
sample_img_path = TRAIN_IMAGES_DIR / f"{sample_stem}.png"

# Đọc ảnh bằng OpenCV và đổi hệ màu từ BGR sang RGB để vẽ bằng Matplotlib
img_bgr = cv2.imread(str(sample_img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_h, img_w = img_rgb.shape[:2]

print(f"Đang xem ảnh: {sample_img_path.name}")
print(f"Kích thước ảnh: {img_w} x {img_h} pixel (Chiều rộng x Chiều cao)")

plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title(f"Ảnh gốc: {sample_img_path.name} ({img_w}x{img_h} px)")
plt.axis('off')
plt.show()


### 7 & 8. Đọc nhãn YOLO tương ứng và giải thích các cột

#### Ý nghĩa định dạng nhãn YOLO:
Mỗi dòng trong file nhãn `.txt` tương ứng với một bounding box của một đối tượng:
```text
class_id cx cy w h
```
- **`class_id`**: Số nguyên từ `0` đến `9`, chỉ vị trí của lớp đối tượng trong danh sách `classes.txt`.
- **`cx` (Center X)**: Tọa độ tâm bounding box theo trục ngang, đã được chuẩn hóa chia cho chiều rộng ảnh ($cx = \frac{X_{center}}{Width} \in [0, 1]$).
- **`cy` (Center Y)**: Tọa độ tâm bounding box theo trục dọc, đã được chuẩn hóa chia cho chiều cao ảnh ($cy = \frac{Y_{center}}{Height} \in [0, 1]$).
- **`w` (Width)**: Chiều rộng của bounding box, đã được chuẩn hóa chia cho chiều rộng ảnh ($w = \frac{Box\_Width}{Width} \in [0, 1]$).
- **`h` (Height)**: Chiều cao của bounding box, đã được chuẩn hóa chia cho chiều cao ảnh ($h = \frac{Box\_Height}{Height} \in [0, 1]$).

> **Lưu ý**: Nhờ chuẩn hóa về khoảng $[0, 1]$, mô hình YOLO có thể học mà không bị phụ thuộc vào kích thước ảnh cụ thể.

In [ ]:
sample_label_path = TRAIN_LABELS_DIR / f"{sample_stem}.txt"

annotations = []
with open(sample_label_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cls_id = int(parts[0])
        cx, cy, w, h = map(float, parts[1:5])
        annotations.append((cls_id, cx, cy, w, h))

print(f"File nhãn: {sample_label_path.name}")
print(f"Số lượng vật thể trong ảnh: {len(annotations)}\n")
for i, (cls_id, cx, cy, w, h) in enumerate(annotations, start=1):
    print(f"Object #{i:02d}: class_id={cls_id} ({classes[cls_id]:<22}) | cx={cx:.4f}, cy={cy:.4f}, w={w:.4f}, h={h:.4f}")


### 9 & 10. Chuyển đổi tọa độ YOLO sang pixel và vẽ Bounding Box lên ảnh

Công thức tính tọa độ pixel góc trên-trái $(x_{min}, y_{min})$ và góc dưới-phải $(x_{max}, y_{max})$:
$$
x_{min} = \left(cx - \frac{w}{2}\right) \times Width, \quad
y_{min} = \left(cy - \frac{h}{2}\right) \times Height
$$
$$
x_{max} = \left(cx + \frac{w}{2}\right) \times Width, \quad
y_{max} = \left(cy + \frac{h}{2}\right) \times Height
$$

*Chú ý: Không làm tròn tọa độ quá sớm trong các phép tính toán học để tránh tích lũy sai số.*

In [ ]:
def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x_min = (cx - w / 2.0) * img_w
    y_min = (cy - h / 2.0) * img_h
    x_max = (cx + w / 2.0) * img_w
    y_max = (cy + h / 2.0) * img_h
    return x_min, y_min, x_max, y_max

# Màu sắc cố định cho từng lớp
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))[:, :3] * 255

annotated_display = img_rgb.copy()

for cls_id, cx, cy, w, h in annotations:
    x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, img_w, img_h)
    # Làm tròn sang số nguyên để vẽ lên ảnh pixel
    x1, y1, x2, y2 = int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))
    
    color = tuple(map(int, colors[cls_id]))
    cv2.rectangle(annotated_display, (x1, y1), (x2, y2), color, 3)
    
    label_text = f"{cls_id}: {classes[cls_id]}"
    # Vẽ nền chữ
    cv2.rectangle(annotated_display, (x1, max(0, y1 - 25)), (x1 + len(label_text) * 11, y1), color, -1)
    cv2.putText(annotated_display, label_text, (x1 + 3, max(15, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)

plt.figure(figsize=(12, 7))
plt.imshow(annotated_display)
plt.title(f"Trực quan hóa nhãn YOLO trên ảnh: {sample_img_path.name}")
plt.axis('off')
plt.show()


### 11 & 12. Thống kê số lượng object từng class và số ảnh có label rỗng

In [ ]:
class_counts = Counter()
empty_label_files = []
total_objects = 0

for lbl_file in train_labels:
    with open(lbl_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    if not lines:
        empty_label_files.append(lbl_file.name)
    else:
        for line in lines:
            cls_id = int(line.split()[0])
            class_counts[cls_id] += 1
            total_objects += 1

print(f"=== THỐNG KÊ TOÀN BỘ TẬP TRAIN ===")
print(f"Tổng số ảnh train:               {len(train_labels)}")
print(f"Số ảnh có nhãn rỗng (background): {len(empty_label_files)} ({len(empty_label_files)/len(train_labels)*100:.1f}%)")
print(f"Tổng số lượng object được gán nhãn: {total_objects}")
print("-" * 45)
for cid in range(len(classes)):
    count = class_counts[cid]
    pct = (count / total_objects * 100) if total_objects > 0 else 0
    print(f"  Lớp {cid:02d} ({classes[cid]:<22}): {count:>4} đối tượng ({pct:>5.1f}%)")


### 13. Vẽ biểu đồ Bar Chart phân bố số lượng object theo từng class

In [ ]:
class_names_ordered = [classes[cid] for cid in range(len(classes))]
counts_ordered = [class_counts[cid] for cid in range(len(classes))]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(classes)), counts_ordered, color='#2b5c8f', edgecolor='black', alpha=0.85)

# Hiển thị giá trị cụ thể trên đầu mỗi cột
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, h + 2, f"{int(h)}",
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(range(len(classes)), class_names_ordered, rotation=35, ha='right')
plt.title("Phân bố số lượng đối tượng theo từng Class trong tập Train", fontsize=14, fontweight='bold')
plt.xlabel("Tên lớp đối tượng (Class Name)", fontsize=12)
plt.ylabel("Số lượng đối tượng (Bounding Boxes)", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
